# Cross-sell & Upsell — fichier financier

Comptage des deals **Upsell** et **Cross-sell** dans `data/export perf sprint.xlsx` (colonnes `upsell` / `cross_sell`), avec répartition par cohorte et par CA.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(pathlib.Path().resolve()))
from financial_sprint_loader import FinancialSprintLoader

fin = FinancialSprintLoader().load()
print(f'{len(fin)} deals chargés')

In [ ]:
# ── comptage global ───────────────────────────────────────────────────────
n_total     = len(fin)
n_upsell    = fin['upsell'].notna().sum()
n_crosssell = fin['cross_sell'].notna().sum()
n_ni        = n_total - n_upsell - n_crosssell   # ni upsell ni cross-sell

print(f"{'Catégorie':<14}{'deals':>8}{'part':>9}{'CA (€)':>14}")
print('-' * 45)
for label, mask in [
    ('Upsell',      fin['upsell'].notna()),
    ('Cross-sell',  fin['cross_sell'].notna()),
    ('Ni/ni',       fin['upsell'].isna() & fin['cross_sell'].isna()),
    ('TOTAL',       np.ones(len(fin), dtype=bool)),
]:
    n  = int(mask.sum())
    ca = fin.loc[mask, 'deal_value_eur'].sum()
    print(f'{label:<14}{n:>8}{100*n/n_total:>8.1f}%{ca:>14,.0f}')

In [ ]:
# ── répartition par cohorte (année) ───────────────────────────────────────
fin = fin.assign(
    is_upsell=fin['upsell'].notna().astype(int),
    is_crosssell=fin['cross_sell'].notna().astype(int),
)
by_year = (fin.groupby('cohorte')[['is_upsell', 'is_crosssell']]
             .sum().rename(columns={'is_upsell': 'Upsell', 'is_crosssell': 'Cross-sell'}))
by_year['deals'] = fin.groupby('cohorte').size()
print(by_year.to_string())

In [ ]:
# ── graphe : upsell vs cross-sell par cohorte ─────────────────────────────
yrs = by_year.index.astype(int).astype(str)
x = np.arange(len(yrs)); w = 0.38

fig, ax = plt.subplots(figsize=(10, 5.5))
fig.patch.set_facecolor('#f4f4f7'); ax.set_facecolor('#fdfdff')

b1 = ax.bar(x - w/2, by_year['Upsell'],     w, color='#4A90D9', label='Upsell', zorder=3)
b2 = ax.bar(x + w/2, by_year['Cross-sell'], w, color='#E67E22', label='Cross-sell', zorder=3)
ax.bar_label(b1, padding=2, fontsize=9, color='#000')
ax.bar_label(b2, padding=2, fontsize=9, color='#000')

ax.set_xticks(x); ax.set_xticklabels(yrs)
ax.grid(axis='y', color='grey', alpha=0.15, linewidth=0.7); ax.set_axisbelow(True)
ax.set_xlabel('Cohorte', color='#000', fontsize=11)
ax.set_ylabel('nombre de deals', color='#000', fontsize=11)
ax.tick_params(colors='#000', labelsize=10)
for spine in ax.spines.values():
    spine.set_edgecolor('#888')
ax.legend(facecolor='#fff', edgecolor='#888', labelcolor='#000', fontsize=10)
ax.set_title(f'Upsell & Cross-sell par cohorte — {n_upsell} upsell · {n_crosssell} cross-sell',
             color='#000', fontsize=12, pad=12)
plt.tight_layout(); plt.show()
#@todo : filter on france, group by client, filter partners

In [ ]:
# ── détail des deals cross-sell & upsell ──────────────────────────────────
detail = fin[fin['upsell'].notna() | fin['cross_sell'].notna()].copy()
detail['type'] = np.where(detail['upsell'].notna(), 'Upsell', 'Cross-sell')
print(detail[['estimate', 'client', 'cohorte', 'type', 'deal_value_eur', 'motion']]
      .sort_values(['type', 'cohorte']).to_string(index=False))